# Thief Detector
## This task tests your Image Processing skills to build a motion detection algorithm that alarms you when you have an unwanted visitor in your home.

## Steps
- 1. Get the live video feed from your webcam
- 2. Fix a scene (the place you want to monitor) and store it as a reference background image
    - Store the first frame as the reference background frame
- 3. For every frame, check if there is any unwanted object inside the scene you are monitoring
    - Use **Background Subtraction** concept (**cv2.absdiff( )**)
        - Subtract the current frame from the reference background image(frame) to see the changes in the scene
        - If there is enormous amount of pixels distrubed in the subtraction result image
            - unwanted visitor (place is unsafe --> alarm the authorities)
        - If there is no enormous amount of pixels distrubed in the subtraction result image
            - no unwanted visitor (place is safe)
- 4. Output the text **"UNSAFE"** in **red** color on the top right of the frame when there is an intruder in the scene.
- 5. Save the live feed
- 6. Submit the (.ipynb) file

## Get live video feed from webcam [10 points]

In [2]:
import cv2
import numpy as np
import datetime

# Open a connection to the default webcam (0 = first camera device)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise IOError("Cannot open webcam. Check that a camera is connected and accessible.")


frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Webcam opened successfully. Resolution: {frame_width}x{frame_height}")

Webcam opened successfully. Resolution: 640x480


## Read first frame, convert to Grayscale and store it as reference background image [10 points]

In [3]:
ret, first_frame = cap.read()

if not ret:
    raise IOError("Failed to read the first frame from the webcam.")

background_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
background_gray = cv2.GaussianBlur(background_gray, (21, 21), 0)

print("Reference background frame captured and converted to grayscale.")

Reference background frame captured and converted to grayscale.


## Compute Absolute Difference between Current and First frame [20 points]

In [4]:
def get_frame_diff(current_frame, first_frame):
   
    current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
    current_gray = cv2.GaussianBlur(current_gray, (21, 21), 0)

    diff = cv2.absdiff(first_frame, current_gray)
    return current_gray, diff

## Apply threshold [5 points]

In [5]:
def get_threshold(diff_image, thresh_val=30):
   
    _, thresh = cv2.threshold(diff_image, thresh_val, 255, cv2.THRESH_BINARY)
    thresh = cv2.dilate(thresh, None, iterations=2)
    return thresh

## Find contours [10 points]

In [6]:
def get_contours(thresh_image):
   
    contours, _ = cv2.findContours(thresh_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours

## Check if contourArea is large and draw rectangle around the object, output "UNSAFE" text in red color [30 points]

In [8]:
MIN_CONTOUR_AREA = 5000

def mark_intrusions(frame, contours, min_area=MIN_CONTOUR_AREA):
    is_unsafe = False

    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        is_unsafe = True
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

    if is_unsafe:
        cv2.putText(frame, "UNSAFE", (frame.shape[1] - 220, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    else:
        cv2.putText(frame, "SAFE", (frame.shape[1] - 200, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    return frame, is_unsafe   # <-- must be at this indentation level (inside the function, after the if/else)

## Display images [10 points]

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter("thief_detector_output.avi", fourcc, 20.0, (frame_width, frame_height))

print("Starting live feed. Press any button in the video window to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame from webcam. Stopping.")
        break

    current_gray, diff = get_frame_diff(frame, background_gray)

    thresh = get_threshold(diff)

    contours = get_contours(thresh)

    annotated_frame, is_unsafe = mark_intrusions(frame.copy(), contours)

    if is_unsafe:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[ALERT] Intruder detected at {timestamp}")

    out.write(annotated_frame)

    cv2.imshow("Thief Detector - Live Feed", annotated_frame)
    cv2.imshow("Threshold Mask", thresh)

    if cv2.waitKey(1) != -1:
        break

Starting live feed. Press any button in the video window to stop.
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[ALERT] Intruder detected at 2026-07-24 19:31:38
[AL

: 

## Release objects [5 points]

In [ ]:
cap.release()
out.release()
cv2.destroyAllWindows()

print("Resources released. Output video saved as 'thief_detector_output.avi'.")